# 00 · Database setup

Creates `data.db` and its schema. Runs once at the start of the project, or
whenever a table has to be rebuilt from scratch.

| Table | Contents | Columns |
|---|---|---|
| `metadata` | WikiArt catalogue: artist, genre, movement | 4 |
| `mfdfa_b1` | MF-DFA, band 6 px to 25% | 197 |
| `mfdfa_b2` | MF-DFA, band 25% to 75% | 197 |
| `mfrenyi_b1` | MF-Rényi, band 6 px to 25% | 183 |
| `mfrenyi_b2` | MF-Rényi, band 25% to 75% | 183 |

Every table is keyed by `painting_id`, the position of the painting in the
dataset.

**No feature extraction happens here.** This notebook only defines the
container; notebooks 01 onwards fill it.

In [ ]:
import sqlite3
import pandas as pd

import db

## Painting catalogue

Only the label columns are requested, in streaming mode, so the images are
never downloaded.

In [ ]:
con = db.connect()
db.create_metadata_table(con)

## Feature tables

One table per method and band. `drop=True` rebuilds from scratch: use it only
when losing the current contents is intended.

In [ ]:
db.create_mfdfa_b1(con, drop=True)
db.create_mfdfa_b2(con, drop=True)
db.create_mfrenyi_b1(con, drop=True)
db.create_mfrenyi_b2(con, drop=True)

## Schema check

Which tables exist, how many columns each one has, and how many rows it holds.

In [ ]:
tables = [f[0] for f in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)]

for t in tables:
    n_cols = len(list(con.execute(f'PRAGMA table_info("{t}")')))
    n_rows = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"{t:<16} {n_cols:>4} columns   {n_rows:>7} rows")

## Catalogue overview

Label distribution before any filtering.

In [ ]:
pd.read_sql_query("""
    SELECT movement, COUNT(*) AS paintings
    FROM metadata
    GROUP BY movement
    ORDER BY paintings DESC
    LIMIT 15
""", con)

In [ ]:
pd.read_sql_query("""
    SELECT
        COUNT(*)                 AS total,
        COUNT(DISTINCT artist)   AS artists,
        COUNT(DISTINCT movement) AS movements,
        COUNT(DISTINCT genre)    AS genres
    FROM metadata
""", con)

## Categories above the 200-painting threshold

These counts define the classification tasks of the experimental chapter.

Check how WikiArt spells its unknown labels before trusting the filters below;
the next query lists them.

In [ ]:
pd.read_sql_query("""
    SELECT artist AS label, COUNT(*) AS paintings FROM metadata
    WHERE artist LIKE '%nknown%'
    GROUP BY artist
    UNION ALL
    SELECT genre, COUNT(*) FROM metadata
    WHERE genre LIKE '%nknown%'
    GROUP BY genre
""", con)

In [ ]:
UNKNOWN_ARTIST = 'Unknown Artist'   # adjust from the query above
UNKNOWN_GENRE   = 'Unknown Genre'

pd.read_sql_query("""
    SELECT 'movement' AS label, COUNT(*) AS categories FROM (
        SELECT movement FROM metadata
        GROUP BY movement HAVING COUNT(*) >= 200
    )
    UNION ALL
    SELECT 'artist', COUNT(*) FROM (
        SELECT artist FROM metadata
        WHERE artist != :unknown_artist
        GROUP BY artist HAVING COUNT(*) >= 200
    )
    UNION ALL
    SELECT 'genre', COUNT(*) FROM (
        SELECT genre FROM metadata
        WHERE genre != :unknown_genre
        GROUP BY genre HAVING COUNT(*) >= 200
    )
""", con, params={"unknown_artist": UNKNOWN_ARTIST,
                    "unknown_genre": UNKNOWN_GENRE})

In [ ]:
con.close()